<a href="https://colab.research.google.com/github/Timang419/deep-learning-for-mortgage-/blob/main/data_cleaning_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import gc
import h5py
import hdf5plugin
from multiprocessing import Pool

path

In [ ]:
BASE_DIR   = '/data/math-deep-learning-course/linc6017/dissertation/dataset'
OUTPUT_DIR = '/data/math-deep-learning-course/linc6017/dissertation/hdf5/full'
os.makedirs(OUTPUT_DIR, exist_ok=True)
N_SHARDS        = 10
SHARD_PATHS     = [os.path.join(OUTPUT_DIR, f'train_shard_{i}.h5') for i in range(N_SHARDS)]
VAL_H5          = os.path.join(OUTPUT_DIR, 'val.h5')
TEST_H5         = os.path.join(OUTPUT_DIR, 'test.h5')


In [ ]:
TRAIN_YEARS     = set(range(1999, 2017))   # 1999–2017
VAL_YEARS       = set(range(2017, 2021))   # 2017–2020
TEST_YEARS      = set(range(2021, 2026))   # 2021-2025

RANDOM_SEED     = 42
DIST_CHUNK_SIZE = 500_000
N_WORKERS       = 4

In [ ]:
ORIG_COLS = [
    'Credit Score', 'First Payment Date', 'First Time Homebuyer Flag',
    'Maturity Date', 'MSA', 'Mortgage Insurance Percentage', 'Number of Units',
    'Occupancy Status', 'Original Combined Loan-to-Value (CLTV)',
    'Original Debt-to-Income (DTI) Ratio', 'Original UPB',
    'Original Loan-to-Value (LTV)', 'Original Interest Rate', 'Channel',
    'Prepayment Penalty Mortgage (PPM) Flag', 'Amortization Type',
    'Property State', 'Property Type', 'Postal Code', 'Loan Sequence Number',
    'Loan Purpose', 'Original Loan Term', 'Number of Borrowers',
    'Seller Name', 'Servicer Name', 'Super Conforming Flag',
    'Pre-HARP Loan Sequence Number', 'Program Indicator', 'HARP Indicator',
    'Property Valuation Method', 'Interest Only (I/O) Indicator',
    'Mortgage Insurance Cancellation Indicator',
]

ORIG_COLS_KEEP = [
    'Credit Score', 'First Payment Date', 'First Time Homebuyer Flag',
    'MSA', 'Mortgage Insurance Percentage', 'Number of Units',
    'Occupancy Status', 'Original Combined Loan-to-Value (CLTV)',
    'Original Debt-to-Income (DTI) Ratio', 'Original UPB',
    'Original Loan-to-Value (LTV)', 'Original Interest Rate', 'Channel',
    'Prepayment Penalty Mortgage (PPM) Flag',
    'Property State', 'Property Type', 'Loan Sequence Number',
    'Loan Purpose', 'Original Loan Term', 'Number of Borrowers',
    'Super Conforming Flag', 'Program Indicator', 'HARP Indicator',
    'Property Valuation Method', 'Interest Only (I/O) Indicator',
    'Mortgage Insurance Cancellation Indicator'
]

ORIG_NUM_COLS = [
    'Credit Score', 'Mortgage Insurance Percentage', 'Number of Units',
    'Original Combined Loan-to-Value (CLTV)', 'Original Debt-to-Income (DTI) Ratio',
    'Original UPB', 'Original Loan-to-Value (LTV)', 'Original Interest Rate',
    'Original Loan Term'
]

SVCG_COLS = [
    'Loan Sequence Number', 'Monthly Reporting Period', 'Current Actual UPB',
    'Current Loan Delinquency Status', 'Loan Age',
    'Remaining Months to Legal Maturity', 'Defect Settlement Date',
    'Modification Flag', 'Zero Balance Code', 'Zero Balance Effective Date',
    'Current Interest Rate', 'Current Non-Interest Bearing UPB',
    'Due Date of Last Paid Installment (DDLPI)', 'MI Recoveries',
    'Net Sales Proceeds', 'Non MI Recoveries', 'Total Expenses', 'Legal Costs',
    'Maintenance and Preservation Costs', 'Taxes and Insurance',
    'Miscellaneous Expenses', 'Actual Loss Calculation', 'Cumulative Modification Cost',
    'Step Modification Flag', 'Payment Deferral Flag',
    'Estimated Loan-to-Value (ELTV)', 'Zero Balance Removal UPB',
    'Delinquent Accrued Interest', 'Delinquency Due to Disaster',
    'Borrower Assistance Status Code', 'Current Month Modification Cost',
    'Interest Bearing UPB',
]

SVCG_COLS_KEEP = [
    'Loan Sequence Number', 'Monthly Reporting Period', 'Current Actual UPB',
    'Current Loan Delinquency Status', 'Loan Age',
    'Remaining Months to Legal Maturity',
    'Modification Flag', 'Zero Balance Code',
    'Current Interest Rate', 'Current Non-Interest Bearing UPB',
    'Due Date of Last Paid Installment (DDLPI)',
    'Step Modification Flag', 'Payment Deferral Flag',
    'Delinquency Due to Disaster',
    'Borrower Assistance Status Code', 'Current Month Modification Cost',
    'Interest Bearing UPB',
]

SVCG_NUM_COLS = [
    'Current Actual UPB', 'Loan Age', 'Remaining Months to Legal Maturity',
    'Current Interest Rate', 'Current Non-Interest Bearing UPB',
    'Current Month Modification Cost', 'Interest Bearing UPB'
]

# ── Categorical definitions ───────────────────────────────────────────────────
KNOWN_STATES = {
    'AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA',
    'GU', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD',
    'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ',
    'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'PR', 'RI', 'SC', 'SD',
    'TN', 'TX', 'UT', 'VA', 'VI', 'VT', 'WA', 'WI', 'WV', 'WY'
}
CATEGORICAL_STATES = sorted(list(KNOWN_STATES)) + ['OTHER']

GLOBAL_CATEGORIES = {
    'First Time Homebuyer Flag'              : ['N', 'Y', '9'],
    'Number of Units'                        : ['1', '2', '3', '4', '99'],
    'Occupancy Status'                       : ['P', 'I', 'S', '9'],
    'Channel'                                : ['R', '9', 'B', 'C', 'T'],
    'Property Type'                          : ['SF', '99', 'CO', 'CP', 'MH', 'PU'],
    'Loan Purpose'                           : ['P', '9', 'C', 'N', 'R'],
    'Borrower_Cat'                           : ['Single', 'Multiple', 'Missing'],
    'Program Indicator'                      : ['9', 'F', 'H', 'R'],
    'Property Valuation Method'              : ['2', '1', '3', '4', '7'],
    'Mortgage Insurance Cancellation Indicator': ['7', 'N', 'Y', '9'],
    'Property State'                         : CATEGORICAL_STATES,
    'Origination Vintage': [str(y) for y in range(1999, 2026)],
}

PREFIX_MAP = {
    'First Time Homebuyer Flag'              : 'FTHB',
    'Number of Units'                        : 'UNITS',
    'Occupancy Status'                       : 'OCC',
    'Channel'                                : 'CHAN',
    'Property Type'                          : 'PROP',
    'Loan Purpose'                           : 'PURP',
    'Borrower_Cat'                           : 'BORR',
    'Program Indicator'                      : 'PROG',
    'Property Valuation Method'              : 'VAL',
    'Mortgage Insurance Cancellation Indicator': 'MI_CANCEL',
    'Property State'                         : 'STATE',
    'Origination Vintage'                    : 'VINT',
}

# Expected current-state dummy columns (states 0-4, absorbing states already filtered)
CURRENT_STATE_DUMMIES = ['state_0', 'state_1', 'state_2', 'state_3', 'state_4']
# 0=Current, 1=D30, 2=D60, 3=D90Plus, 4=Foreclosure

In [ ]:
def clean_orig_pandas(df, global_mi_median, global_dti_median):
    """Clean one origination dataframe. Medians must be pre-computed globally."""

    # 1. Drop sentinel credit scores and LTV
    df = df[(df['Credit Score'] != 9999) & (df['Credit Score'].notnull())]
    df = df[(df['Original Loan-to-Value (LTV)'] != 999) & (df['Original Loan-to-Value (LTV)'].notnull())]

    # 2. First Payment Date → Vintage year
    df['First Payment Date'] = pd.to_datetime(
        df['First Payment Date'].astype(str), format='%Y%m', errors='coerce')
    df['Origination Vintage'] = df['First Payment Date'].dt.year
    df['Origination Vintage'] = df['Origination Vintage'].astype(str)
    df = df.drop(columns=['First Payment Date'])

    # 3. MSA → binary indicator
    df['MSA'] = (~df['MSA'].isna()).astype(np.int8)

    # 4. Mortgage Insurance Percentage (sentinel 999 → global median)
    df['MI_OUT_OF_RANGE'] = (df['Mortgage Insurance Percentage'] == 999).astype(np.int8)
    df['Mortgage Insurance Percentage'] = df['Mortgage Insurance Percentage'].mask(
        df['Mortgage Insurance Percentage'] == 999, global_mi_median
    )

    # 5. CLTV (sentinel 999 → LTV fallback)
    df['CLTV_MISSING'] = (df['Original Combined Loan-to-Value (CLTV)'] == 999).astype(np.int8)
    df['CLTV_clean'] = df['Original Combined Loan-to-Value (CLTV)'].mask(
        df['Original Combined Loan-to-Value (CLTV)'] == 999,
        df['Original Loan-to-Value (LTV)']
    )
    df = df.drop(columns=['Original Combined Loan-to-Value (CLTV)'])

    # 6. DTI Ratio (sentinel 999 → global median)
    df['DTI_MISSING'] = (df['Original Debt-to-Income (DTI) Ratio'] == 999).astype(np.int8)
    df['Original Debt-to-Income (DTI) Ratio'] = df['Original Debt-to-Income (DTI) Ratio'].mask(
        df['Original Debt-to-Income (DTI) Ratio'] == 999, global_dti_median
    )

    # 7. Binary flags
    df['Prepayment Penalty Mortgage (PPM) Flag'] = (
        df['Prepayment Penalty Mortgage (PPM) Flag'].astype(str).str.strip().str.upper() == 'Y').astype(np.int8)
    df['Super Conforming Flag'] = (
        df['Super Conforming Flag'].astype(str).str.strip() == 'Y').astype(np.int8)
    df['HARP Indicator'] = (
        df['HARP Indicator'].astype(str).str.strip() == 'Y').astype(np.int8)
    df['Interest Only (I/O) Indicator'] = (
        df['Interest Only (I/O) Indicator'].astype(str).str.strip() == 'Y').astype(np.int8)

    if 'Pre-HARP Loan Sequence Number' in df.columns:
        df = df.drop(columns=['Pre-HARP Loan Sequence Number'])

    # 8. Borrower count → ordinal category
    df['Number of Borrowers'] = pd.to_numeric(df['Number of Borrowers'], errors='coerce')
    df['Borrower_Cat'] = 'Missing'
    df['Borrower_Cat'] = df['Borrower_Cat'].mask(df['Number of Borrowers'] == 1, 'Single')
    df['Borrower_Cat'] = df['Borrower_Cat'].mask(
        (df['Number of Borrowers'] >= 2) & (df['Number of Borrowers'] <= 10), 'Multiple')
    df = df.drop(columns=['Number of Borrowers'])

    # 9. Property State catch-all
    if 'Property State' in df.columns:
        clean_states = df['Property State'].astype(str).str.strip().str.upper()
        df['Property State'] = np.where(clean_states.isin(KNOWN_STATES), clean_states, 'OTHER')

    # 10. Apply global categorical schema (ensures consistent columns across cohorts)
    for col, cats in GLOBAL_CATEGORIES.items():
        if col in df.columns:
            df[col] = pd.Categorical(df[col].astype(str).str.strip(), categories=cats)

    cols_to_encode = [c for c in GLOBAL_CATEGORIES.keys() if c in df.columns]
    for col in cols_to_encode:
        assert df[col].notna().all(), f"CRITICAL: {col} has unexpected NaN after categorical mapping!"

    prefixes = [PREFIX_MAP[c] for c in cols_to_encode]

    # KEY: drop_first=False — keep ALL dummies.
    # L2 regularisation (weight decay) in all downstream models resolves
    # the underdetermined parameter space arising from perfect multicollinearity.
    # This follows Sirignano & Giesecke (2019) footnote 20 convention.
    df = pd.get_dummies(df, columns=cols_to_encode, prefix=prefixes,
                        drop_first=False, dtype=np.int8)
    return df

In [ ]:
def clean_svcg_pandas(df):
    """Clean one servicing dataframe chunk."""

    # ── Modification Flag ──────────────────────────────────────────────────────
    # Keep ALL dummies including baseline 'N' — no drop
    df['Modification Flag'] = df['Modification Flag'].fillna('N')
    df = pd.get_dummies(df, columns=['Modification Flag'], prefix='mod_flag',
                        drop_first=False, dtype=np.int8)
    for col in ['mod_flag_N', 'mod_flag_Y', 'mod_flag_P']:
        if col not in df.columns:
            df[col] = np.int8(0)

    # ── DDLPI → Months Since Paid ──────────────────────────────────────────────
    df['Due Date of Last Paid Installment (DDLPI)'] = pd.to_datetime(
        df['Due Date of Last Paid Installment (DDLPI)'], format='%Y%m', errors='coerce')
    df['Monthly Reporting Period'] = pd.to_datetime(
        df['Monthly Reporting Period'], format='%Y%m', errors='coerce')
    df['Months Since Paid'] = (
        (df['Monthly Reporting Period'].dt.year  - df['Due Date of Last Paid Installment (DDLPI)'].dt.year)  * 12 +
        (df['Monthly Reporting Period'].dt.month - df['Due Date of Last Paid Installment (DDLPI)'].dt.month)
    ).fillna(0)
    # Convert Monthly Reporting Period to integer YYYYMM for HDF5 storage
    df['Reporting_Period_Int'] = (
        df['Monthly Reporting Period'].dt.year * 100 +
        df['Monthly Reporting Period'].dt.month
    ).fillna(0).astype(int)
    df.drop(columns=['Due Date of Last Paid Installment (DDLPI)',
                     'Monthly Reporting Period'], inplace=True)

    # ── Step Modification Flag ─────────────────────────────────────────────────
    # Keep ALL dummies including 'None' baseline
    df['Step Modification Flag'] = df['Step Modification Flag'].fillna('None')
    df = pd.get_dummies(df, columns=['Step Modification Flag'], prefix='step_mod',
                        drop_first=False, dtype=np.int8)
    for col in ['step_mod_None', 'step_mod_N', 'step_mod_Y']:
        if col not in df.columns:
            df[col] = np.int8(0)

    # ── Payment Deferral Flag ──────────────────────────────────────────────────
    # Keep ALL dummies including 'N' baseline
    df['Payment Deferral Flag'] = df['Payment Deferral Flag'].fillna('N')
    df = pd.get_dummies(df, columns=['Payment Deferral Flag'], prefix='deferral',
                        drop_first=False, dtype=np.int8)
    for col in ['deferral_N', 'deferral_Y', 'deferral_P']:
        if col not in df.columns:
            df[col] = np.int8(0)

    # ── Delinquency Due to Disaster ────────────────────────────────────────────
    df['Delinquency Due to Disaster'] = (df['Delinquency Due to Disaster'] == 'Y').astype(np.int8)

    # ── Borrower Assistance Status Code ───────────────────────────────────────
    # Keep ALL dummies including 'None' baseline
    df['Borrower Assistance Status Code'] = df['Borrower Assistance Status Code'].fillna('None')
    df = pd.get_dummies(df, columns=['Borrower Assistance Status Code'], prefix='assist',
                        drop_first=False, dtype=np.int8)
    for col in ['assist_None', 'assist_F', 'assist_R', 'assist_T']:
        if col not in df.columns:
            df[col] = np.int8(0)

    # ── Cumulative Modification Cost (engineered, leak-free) ──────────────────
    df['Current Month Modification Cost'] = df['Current Month Modification Cost'].fillna(0)
    df = df.sort_values(by=['Loan Sequence Number', 'Reporting_Period_Int'])
    df['Cumulative_Mod_Cost_Engineered'] = (
        df.groupby('Loan Sequence Number')['Current Month Modification Cost'].cumsum()
    )

    _dq_h = pd.to_numeric(df['Current Loan Delinquency Status'],
                          errors='coerce').fillna(0)
    df['_d30']  = (_dq_h == 1).astype(np.float32)
    df['_d60']  = (_dq_h == 2).astype(np.float32)
    df['_d90p'] = (_dq_h >= 3).astype(np.float32)
    _g = df.groupby('Loan Sequence Number', sort=False)
    df['D30_count_12m']     = (_g['_d30'].transform(
        lambda x: x.shift(1).rolling(12, min_periods=0).sum())
        .fillna(0).clip(0, 12).astype(np.int8))
    df['D60_count_12m']     = (_g['_d60'].transform(
        lambda x: x.shift(1).rolling(12, min_periods=0).sum())
        .fillna(0).clip(0, 12).astype(np.int8))
    df['D90plus_count_12m'] = (_g['_d90p'].transform(
        lambda x: x.shift(1).rolling(12, min_periods=0).sum())
        .fillna(0).clip(0, 12).astype(np.int8))
    df.drop(columns=['_d30', '_d60', '_d90p'], inplace=True)
    del _dq_h, _g

    # ── State construction ─────────────────────────────────────────────────────
    zbc    = df['Zero Balance Code'].fillna('').astype(str).str.zfill(2)
    dq     = df['Current Loan Delinquency Status'].astype(str).str.strip()
    dq_num = pd.to_numeric(dq, errors='coerce')

    conditions = [
        zbc.isin(['15', '16', '96']),           # Right-censored  → drop
        zbc.isin(['01', '02', '03']),            # Paid Off        → 6
        (dq == 'RA') | (zbc == '09'),            # REO             → 5
        dq_num == 0,                             # Current         → 0
        dq_num == 1,                             # D30             → 1
        dq_num == 2,                             # D60             → 2
        (dq_num >= 3) & (dq_num <= 5),           # D90+            → 3
        dq_num >= 6                              # Foreclosure     → 4
    ]
    choices = [np.nan, 6, 5, 0, 1, 2, 3, 4]

    df['state']      = np.select(conditions, choices, default=np.nan)
    df['next_state'] = df.groupby('Loan Sequence Number')['state'].shift(-1)

    # Drop right-censored rows and rows with no next observation
    df = df.dropna(subset=['next_state'])
    # Drop rows where current state is absorbing (REO=5 or Paid Off=6)
    df = df[df['state'] < 5]

    # BUG FIX: convert to int AFTER absorbing state filter (RA rows gone now)
    df['state']      = df['state'].astype(np.int8)
    df['next_state'] = df['next_state'].astype(np.int8)

    # DQ status: safe to convert now since all RA rows are filtered out
    df['Current Loan Delinquency Status'] = pd.to_numeric(
        df['Current Loan Delinquency Status'], errors='coerce'
    ).fillna(0).astype(np.int8)

    # Drop Zero Balance Code — would leak outcome into features
    df.drop(columns=['Zero Balance Code'], inplace=True)

    # ── Current state OHE ──────────────────────────────────────────────────────
    # Keep ALL 5 dummies (states 0-4). No drop — L2 regularisation handles
    # the underdetermined parameter space.
    df = pd.get_dummies(df, columns=['state'], prefix='state',
                        drop_first=False, dtype=np.int8)
    for col in CURRENT_STATE_DUMMIES:
        if col not in df.columns:
            df[col] = np.int8(0)

    df=df.dropna()

    return df

global median

In [ ]:
def append_to_hdf5(hdf5_path, X_np, y_np, loan_ids_np, col_names=None):
    """
    Append rows to an HDF5 file. Creates resizable datasets on first call.
    loan_ids_np is converted to object dtype for h5py vlen string compatibility.
    """
    n_new          = X_np.shape[0]
    dt_str         = h5py.special_dtype(vlen=str)
    loan_ids_bytes = np.array(loan_ids_np, dtype=object)   # fix: avoid dtype('<U12') error

    with h5py.File(hdf5_path, 'a') as f:
        if 'X' not in f:
            f.create_dataset('X', data=X_np.astype(np.float32),
                             maxshape=(None, X_np.shape[1]),
                             chunks=(10_000, X_np.shape[1]),
                             **hdf5plugin.LZ4())
            f.create_dataset('y', data=y_np.astype(np.int8),
                             maxshape=(None,),
                             chunks=(10_000,),
                             **hdf5plugin.LZ4())
            f.create_dataset('loan_ids', data=loan_ids_bytes,
                             maxshape=(None,),
                             dtype=dt_str,
                             chunks=(min(10_000, n_new),))
            if col_names is not None:
                f['X'].attrs['col_names'] = col_names
        else:
            n_existing = f['X'].shape[0]
            n_total    = n_existing + n_new
            f['X'].resize((n_total, X_np.shape[1]))
            f['X'][n_existing:] = X_np.astype(np.float32)
            f['y'].resize((n_total,))
            f['y'][n_existing:] = y_np.astype(np.int8)
            f['loan_ids'].resize((n_total,))
            f['loan_ids'][n_existing:] = loan_ids_bytes

In [ ]:
def _scan_one_orig(orig_file):
    """Worker: scan one ORIG file and return (mi_series, dti_series)."""
    df = pd.read_csv(orig_file, sep='|', header=None, names=ORIG_COLS,
                     usecols=['Mortgage Insurance Percentage',
                              'Original Debt-to-Income (DTI) Ratio'],
                     dtype=str)
    df['Mortgage Insurance Percentage']       = pd.to_numeric(df['Mortgage Insurance Percentage'], errors='coerce')
    df['Original Debt-to-Income (DTI) Ratio'] = pd.to_numeric(df['Original Debt-to-Income (DTI) Ratio'], errors='coerce')
    mi  = df.loc[df['Mortgage Insurance Percentage']       != 999, 'Mortgage Insurance Percentage'].values
    dti = df.loc[df['Original Debt-to-Income (DTI) Ratio'] != 999, 'Original Debt-to-Income (DTI) Ratio'].values
    return mi, dti

In [ ]:
def compute_global_medians():
    """
    Scan all ORIG files in parallel to compute global MI% and DTI medians.
    Uses Pool with N_WORKERS processes — each worker reads one ORIG file.
    """
    print("\n" + "=" * 60)
    print("PASS 0: Computing global medians (parallel)")
    print("=" * 60)

    all_orig_files = sorted(glob.glob(os.path.join(BASE_DIR, '*/historical_data_????Q?.txt')))
    print(f"  Found {len(all_orig_files)} ORIG files — scanning with {N_WORKERS} workers...")

    with Pool(processes=N_WORKERS) as pool:
        results = pool.map(_scan_one_orig, all_orig_files)

    mi_all  = np.concatenate([r[0] for r in results])
    dti_all = np.concatenate([r[1] for r in results])

    global_mi_median  = float(np.median(mi_all))
    global_dti_median = float(np.median(dti_all))
    print(f"\n  Global MI median:  {global_mi_median}")
    print(f"  Global DTI median: {global_dti_median}")
    return global_mi_median, global_dti_median

In [ ]:
def main():
    print("=" * 60)
    print("Script 1 (Full): Clean + Split + Shuffle + Shard")
    print("=" * 60)
    print(f"BASE_DIR:   {BASE_DIR}")
    print(f"OUTPUT_DIR: {OUTPUT_DIR}")
    print(f"N_SHARDS:   {N_SHARDS}")
    print(f"N_WORKERS:  {N_WORKERS}")

    # Remove existing output files to start fresh
    for path in SHARD_PATHS + [VAL_H5, TEST_H5]:
        if os.path.exists(path):
            os.remove(path)
            print(f"  Removed: {os.path.basename(path)}")

    # ── Pass 0: global medians (parallel) ─────────────────────────────────────
    global_mi_median, global_dti_median = compute_global_medians()

    # ── Discover cohorts ───────────────────────────────────────────────────────
    orig_files = sorted(glob.glob(os.path.join(BASE_DIR, '*/historical_data_????Q?.txt')))
    cohorts = [
        (os.path.dirname(f),
         os.path.basename(f).replace('historical_data_', '').replace('.txt', ''))
        for f in orig_files
    ]
    print(f"\nFound {len(cohorts)} cohorts.")

    col_names_written = False
    shard_counter     = 0
    rng               = np.random.default_rng(RANDOM_SEED)

    # ── Main loop: one cohort at a time ───────────────────────────────────────
    # Sequential — required because all cohorts share the same 10 shard files.
    # Concurrent writes to the same HDF5 file are not safe without locking.
    for year_dir, cohort in cohorts:
        print(f"\n{'=' * 60}")
        print(f"Processing Cohort: {cohort}")
        print(f"{'=' * 60}")

        orig_file = os.path.join(year_dir, f'historical_data_{cohort}.txt')
        svcg_file = os.path.join(year_dir, f'historical_data_time_{cohort}.txt')

        if not os.path.exists(svcg_file):
            print(f"  [!] Skipping {cohort}: no matching SVCG file.")
            continue

        # ── Step A: Load & clean ORIG ──────────────────────────────────────────
        print("  -> Loading and cleaning ORIG...")
        df_orig_raw = pd.read_csv(
            orig_file, sep='|', header=None,
            names=ORIG_COLS, usecols=ORIG_COLS_KEEP, dtype=str
        )
        for col in ORIG_NUM_COLS:
            df_orig_raw[col] = pd.to_numeric(df_orig_raw[col], errors='coerce')

        df_orig_cleaned = clean_orig_pandas(df_orig_raw, global_mi_median, global_dti_median)
        del df_orig_raw; gc.collect()

        # ── Step B: Load FULL SVCG (no chunking) ──────────────────────────────
        # Loading in full ensures groupby cumsum and shift(-1) see each loan's
        # complete history — eliminates boundary leakage bug.
        print("  -> Loading full SVCG (no chunking)...")
        svcg_raw = pd.read_csv(
            svcg_file, sep='|', header=None,
            names=SVCG_COLS, usecols=SVCG_COLS_KEEP,
            dtype=str
        )
        for col in SVCG_NUM_COLS:
            svcg_raw[col] = pd.to_numeric(svcg_raw[col], errors='coerce')
        print(f"     SVCG rows loaded: {len(svcg_raw):,}")

        # ── Step C: Clean SVCG ────────────────────────────────────────────────
        print("  -> Cleaning SVCG...")
        svcg_cleaned = clean_svcg_pandas(svcg_raw)
        del svcg_raw; gc.collect()

        # ── Step D: Merge ─────────────────────────────────────────────────────
        print("  -> Merging with ORIG...")
        merged = svcg_cleaned.merge(df_orig_cleaned, on='Loan Sequence Number', how='inner')
        del svcg_cleaned, df_orig_cleaned; gc.collect()

        if merged.empty:
            print("  [!] Empty merged dataframe — skipping cohort.")
            del merged; gc.collect()
            continue

        print(f"     Merged rows: {len(merged):,}")

        # ── Step E: Extract arrays ─────────────────────────────────────────────
        loan_ids_np  = merged['Loan Sequence Number'].values.astype(str)
        drop_cols    = ['Loan Sequence Number', 'next_state']
        feature_cols = [c for c in merged.columns if c not in drop_cols]
        X_np = merged[feature_cols].to_numpy(dtype=np.float32)
        y_np         = merged['next_state'].values.astype(np.int8)
        del merged; gc.collect()

        # ── Step F: Split by reporting year ───────────────────────────────────
        rp_idx   = feature_cols.index('Reporting_Period_Int')
        rep_year = X_np[:, rp_idx].astype(int) // 100   # YYYYMM → YYYY

        train_mask = np.isin(rep_year, list(TRAIN_YEARS))
        val_mask   = np.isin(rep_year, list(VAL_YEARS))
        test_mask  = np.isin(rep_year, list(TEST_YEARS))

        unassigned = (~train_mask & ~val_mask & ~test_mask).sum()
        if unassigned > 0:
            print(f"  WARNING: {unassigned:,} rows outside defined year ranges — dropped.")

        print(f"     Train rows: {train_mask.sum():,} | "
              f"Val rows: {val_mask.sum():,} | "
              f"Test rows: {test_mask.sum():,}")

        # ── Step G: Val and test → append directly ────────────────────────────
        if val_mask.sum() > 0:
            append_to_hdf5(VAL_H5,
                           X_np[val_mask], y_np[val_mask], loan_ids_np[val_mask],
                           col_names=feature_cols if not col_names_written else None)

        if test_mask.sum() > 0:
            append_to_hdf5(TEST_H5,
                           X_np[test_mask], y_np[test_mask], loan_ids_np[test_mask],
                           col_names=feature_cols if not col_names_written else None)

        # ── Step H: Train → shuffle → round-robin to shards ──────────────────
        if train_mask.sum() > 0:
            X_train   = X_np[train_mask]
            y_train   = y_np[train_mask]
            ids_train = loan_ids_np[train_mask]

            # Random shuffle within this cohort's train rows
            perm      = rng.permutation(len(y_train))
            X_train   = X_train[perm]
            y_train   = y_train[perm]
            ids_train = ids_train[perm]

            # Distribute 500k chunks round-robin across shards
            n_train = len(y_train)
            for start in range(0, n_train, DIST_CHUNK_SIZE):
                end      = min(start + DIST_CHUNK_SIZE, n_train)
                shard_id = shard_counter % N_SHARDS

                append_to_hdf5(
                    SHARD_PATHS[shard_id],
                    X_train[start:end],
                    y_train[start:end],
                    ids_train[start:end],
                    col_names=feature_cols if not col_names_written else None
                )

                if not col_names_written:
                    col_names_written = True
                    print(f"  Schema written: {len(feature_cols)} features")

                shard_counter += 1

            del X_train, y_train, ids_train

        del X_np, y_np, loan_ids_np; gc.collect()
        print(f"  -> Finished {cohort}. Shard counter: {shard_counter}")

    # ── Summary ────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("All cohorts processed.")
    print(f"  Total round-robin chunks assigned: {shard_counter}")

    for i, path in enumerate(SHARD_PATHS):
        if os.path.exists(path):
            with h5py.File(path, 'r') as f:
                print(f"  shard_{i}: {f['X'].shape[0]:,} rows")

    for label, path in [('val', VAL_H5), ('test', TEST_H5)]:
        if os.path.exists(path):
            with h5py.File(path, 'r') as f:
                print(f"  {label}:    {f['X'].shape[0]:,} rows")

    print("=" * 60)


if __name__ == '__main__':
    main()